## 训练决策树 ##

In [ ]:
## 训练决策树 ##

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn import tree
import matplotlib.pyplot as plt
from joblib import dump, load  # 导入joblib用于保存和加载模型

# 1. 动态创建数据
data = {
    'feature1': [5.1, 4.9, 6.0, 6.3, 7.2, 6.8],
    'feature2': [3.5, 3.0, 2.7, 2.9, 3.6, 3.0],
    'feature3': [1.4, 1.4, 5.1, 5.6, 6.1, 5.5],
    'feature4': [0.2, 0.2, 1.6, 1.8, 2.5, 2.1],
    'label': [0, 0, 1, 1, 2, 2]
}

# 2. 将数据转换为DataFrame
df = pd.DataFrame(data)

# 3. 将DataFrame保存为CSV文件
file_path = 'dynamic_data.csv'  # 文件名
df.to_csv(file_path, index=False)  # 保存为CSV文件，不保存索引

print(f"CSV文件已创建并保存到: {file_path}")

# 4. 从CSV文件加载数据
loaded_data = pd.read_csv(file_path)

# 5. 分离特征和标签
X = loaded_data.iloc[:, :-1]  # 特征（所有列除了最后一列）
y = loaded_data.iloc[:, -1]   # 标签（最后一列）

# 6. 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("训练集特征：")
print(X_train)
print("训练集标签：")
print(y_train)

# 7. 创建决策树分类器
clf = DecisionTreeClassifier(random_state=42)

# 8. 训练模型
clf.fit(X_train, y_train)

# 9. 使用训练好的模型进行预测
y_pred = clf.predict(X_train)

# 10. 计算准确率
accuracy = accuracy_score(y_train, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# 11. 打印分类报告
print("Classification Report:")
print(classification_report(y_train, y_pred))

# 12. 打印混淆矩阵
print("Confusion Matrix:")
print(confusion_matrix(y_train, y_pred))

# 13. 可视化决策树
plt.figure(figsize=(12, 8))
tree.plot_tree(clf, filled=True, feature_names=X.columns, class_names=[str(i) for i in np.unique(y)])
plt.show()

# 14. 保存模型到文件
model_file_path = 'decision_tree_model.joblib'  # 模型文件名
dump(clf, model_file_path)  # 使用joblib保存模型
print(f"模型已保存为: {model_file_path}")

# 15. 加载模型并测试
loaded_model = load(model_file_path)  # 加载模型
y_pred_loaded = loaded_model.predict(X_train)  # 使用加载的模型进行预测
accuracy_loaded = accuracy_score(y_train, y_pred_loaded)  # 计算准确率
print(f"加载模型的准确率: {accuracy_loaded:.2f}")

## 安装onnx依赖库 ##

In [ ]:
!pip install skl2onnx onnx onnxruntime

## 转成安卓专用的onnx模型

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# 定义输入数据的类型
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]

# 将模型转换为ONNX格式
onnx_model = convert_sklearn(clf, initial_types=initial_type)

# 保存ONNX模型
with open("decision_tree_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

## 测试onnx模型 ##

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

# 加载和验证ONNX模型
model_path = 'decision_tree_model.onnx'
onnx_model = onnx.load(model_path)
onnx.checker.check_model(onnx_model)
print("ONNX模型验证通过")

# 准备测试数据
test_data = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.2, 3.4, 5.4, 2.3],
    # 添加更多测试样本
], dtype=np.float32)
print("测试数据形状:", test_data.shape)

# 创建ONNX运行时会话
session = ort.InferenceSession(model_path)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# 进行推理
predictions = session.run([output_name], {input_name: test_data})
print("预测结果:", predictions)

# 解释结果（假设是分类任务且输出为概率）
predicted_probabilities = predictions[0]
# predicted_classes = np.argmax(predicted_probabilities, axis=1)
# print("预测类别:", predicted_classes)
